In [29]:
import requests

EURLEX_WS_URL = 'https://eur-lex.europa.eu/EURLexWebService'

soap_xml = '''<soap:Envelope xmlns:soap="http://www.w3.org/2003/05/soap-envelope" xmlns:sear="http://eur-lex.europa.eu/search">
  <soap:Header>
    <wsse:Security xmlns:wsse="http://docs.oasis-open.org/wss/2004/01/oasis-200401-wss-wssecurity-secext-1.0.xsd" soap:mustUnderstand="true">
      <wsse:UsernameToken xmlns:wsu="http://docs.oasis-open.org/wss/2004/01/oasis-200401-wss-wssecurity-utility-1.0.xsd" wsu:Id="UsernameToken-1">
        <wsse:Username>n00mbdci</wsse:Username>
        <wsse:Password Type="http://docs.oasis-open.org/wss/2004/01/oasis-200401-wss-username-token-profile-1.0#PasswordText">vJgHkuHuFU3</wsse:Password>
      </wsse:UsernameToken>
    </wsse:Security>
  </soap:Header>
  <soap:Body>
    <sear:searchRequest>
      <sear:expertQuery><![CDATA[SELECT CELLAR_ID, DN, TI, CI, LB_ART, TI_SHORT, FM, CT, DD, PROC_NUM, FM, TT, I1, I2, ECLI
 WHERE
(CT = "CONC" OR "ENTR" OR "EXCL" OR "MERG" OR "POSI" OR "PRAT")
AND
(DTS_SUBDOM = "EU_CASE_LAW")
AND
(RJ ~ "European Community" OR RJ ~ "European Union" )
AND
( TI ~ competition OR TI ~ cartel OR TI ~ "dominant position"
OR TE ~ competition OR TE ~ cartel OR TE ~ "dominant position")]]></sear:expertQuery>
      <sear:page>1</sear:page>
      <sear:pageSize>5</sear:pageSize>
      <sear:searchLanguage>en</sear:searchLanguage>
      <sear:showDocumentsAvailableIn>en,de</sear:showDocumentsAvailableIn>
    </sear:searchRequest>
  </soap:Body>
</soap:Envelope>'''

headers = {'Content-Type': 'application/soap+xml; charset=utf-8', 'Accept': 'application/xml'}
resp = requests.post(EURLEX_WS_URL, data=soap_xml.encode('utf-8'), headers=headers, timeout=60)
print(resp.status_code)
print(resp.text)

with open('data/raw/eurlex_response.xml', 'w', encoding='utf-8') as f:
    f.write(resp.text)
print('Response saved to data/raw/eurlex_response.xml')


200
<?xml version='1.0' encoding='UTF-8'?><S:Envelope xmlns:env="http://www.w3.org/2003/05/soap-envelope" xmlns:S="http://www.w3.org/2003/05/soap-envelope"><env:Header/><S:Body><searchResults xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns:searchLayerHelper="xalan//eu.europa.ec.op.searchlayer.domain.util.SearchLayerHelper" xmlns="http://eur-lex.europa.eu/search" xsi:schemaLocation="http://eur-lex.europa.eu/search https://eur-lex.europa.eu/eurlex-ws?xsd=3">
   <numhits>5</numhits>
   <totalhits>1355</totalhits>
   <page>1</page>
   <language>en</language>
   <result>
      <reference>eng_cellar:ba2d7353-f3e9-4826-b04a-cc79d6cdb8d4_en</reference>
      <rank>1</rank>
      <content>
         <DTS_SUBDOM>ALL_ALL</DTS_SUBDOM>
         <DTS_SUBDOM>EU_LAW_ALL</DTS_SUBDOM>
         <DTS_SUBDOM>EU_CASE_LAW</DTS_SUBDOM>
         <NOTICE>
            <EXPRESSION>
               <EXPRESSION_TITLE>
                  <LANG>en</LANG>
               </EXPRESSION_TITLE>
            </EXPRE

In [30]:
import xml.etree.ElementTree as ET
import pandas as pd

NS = {'s': 'http://eur-lex.europa.eu/search'}
SNS = '{http://eur-lex.europa.eu/search}'

root = ET.fromstring(resp.text)


def get_first_text(node, xpath):
    if node is None:
        return ''
    el = node.find(xpath, NS)
    return el.text.strip() if el is not None and el.text else ''


def get_cellar_id_from_work(work_node):
    if work_node is None:
        return ''
    for uri in work_node.findall(f'{SNS}URI'):
        type_el = uri.find(f'{SNS}TYPE')
        if type_el is not None and type_el.text == 'cellar':
            return get_first_text(uri, 's:IDENTIFIER')
    return ''


def get_ct_codes(work_node):
    if work_node is None:
        return ''
    codes = []
    for sm in work_node.findall(f'{SNS}RESOURCE_LEGAL_IS_ABOUT_SUBJECT-MATTER'):
        for child in sm:
            id_el = child.find(f'{SNS}IDENTIFIER')
            if id_el is not None and id_el.text:
                code = id_el.text.strip()
                if code not in codes:
                    codes.append(code)
    return '|'.join(codes)


def derive_court_level_from_celex(celex_id):
    if not celex_id:
        return ''
    if 'CJ' in celex_id:
        return 'court_of_justice'
    if 'TJ' in celex_id:
        return 'general_court'
    if 'FJ' in celex_id:
        return 'civil_service_tribunal'
    return ''


def parse_result_to_row(result_node):
    notice = result_node.find(f'.//{SNS}NOTICE')
    if notice is None:
        return None

    work = notice.find(f'{SNS}WORK')
    expression = notice.find(f'{SNS}EXPRESSION')
    manifestation = notice.find(f'{SNS}MANIFESTATION')

    cellar_id = get_cellar_id_from_work(work)
    celex_id = get_first_text(work, 's:ID_CELEX/s:VALUE')
    title = get_first_text(expression, 's:EXPRESSION_TITLE/s:VALUE')
    document_date = get_first_text(work, 's:WORK_DATE_DOCUMENT/s:VALUE')
    court_level = derive_court_level_from_celex(celex_id)
    resource_type_code = get_first_text(work, 's:WORK_HAS_RESOURCE-TYPE/s:IDENTIFIER')
    treaty_code = get_first_text(work, 's:RESOURCE_LEGAL_BASED_ON_CONCEPT_TREATY/s:IDENTIFIER')
    ct_codes = get_ct_codes(work)
    case_law_parties_raw = get_first_text(manifestation, 's:MANIFESTATION_CASE-LAW_PARTIES/s:VALUE')

    return {
        'cellar_id': cellar_id,
        'celex_id': celex_id,
        'title': title,
        'document_date': document_date,
        'court_level': court_level,
        'resource_type_code': resource_type_code,
        'treaty_code': treaty_code,
        'ct_codes': ct_codes,
        'case_law_parties_raw': case_law_parties_raw,
    }


rows = []
for result in root.iter(f'{SNS}result'):
    row = parse_result_to_row(result)
    if row is not None:
        rows.append(row)

columns = ['cellar_id', 'celex_id', 'title', 'document_date', 'court_level',
           'resource_type_code', 'treaty_code', 'ct_codes', 'case_law_parties_raw']

df = pd.DataFrame(rows, columns=columns)
print(df)

output_path = 'data/processed/cjeu_cases.csv'
df.to_csv(output_path, index=False, encoding='utf-8')
print(f'Saved {len(df)} records to {output_path}')


                              cellar_id     celex_id title document_date  \
0  ba2d7353-f3e9-4826-b04a-cc79d6cdb8d4  61989TJ0068          1992-03-10   
1  ba1070bd-c40d-4171-88df-8578a05e9d17  61994CJ0068          1998-03-31   
2  b6ccae39-9aa5-4253-953a-22b37892e1ff  62002TJ0038          2005-10-25   
3  b6ccb3e5-fa1a-4486-a0d6-086a4a1255d0  61989TJ0065          1993-04-01   
4  4073c8b0-f562-488f-a435-768f4efae3b2  61976CJ0085          1979-02-13   

        court_level resource_type_code treaty_code             ct_codes  \
0     general_court               JUDG   TEEC_1957  CONC|ENTR|POSI|PRAT   
1  court_of_justice               JUDG   TEEC_1957       POSI|CONC|ENTR   
2     general_court               JUDG   TEEC_1957       CONC|ENTR|PRAT   
3     general_court               JUDG   TEEC_1957  ENTR|POSI|CONC|EXCL   
4  court_of_justice               JUDG   TEEC_1957  EXCL|POSI|ENTR|CONC   

                                case_law_parties_raw  
0  In Cases T-68/89, Società Italiana